# Epoch AI datasets — visual overview

Charts over the cleaned Epoch AI data in `data/epoch_ai/` (raw kept in
`raw_data/epoch_ai/`). Data © Epoch AI, CC-BY — see `datasources/epoch_ai.csv` for
per-dataset citations. The last cell catalogs every cleaned file.

In [1]:
from pathlib import Path

import polars as pl
import plotly.express as px

DATA = Path("../data/epoch_ai")


def load(name):
    return pl.read_csv(DATA / name, infer_schema_length=20000, ignore_errors=True, truncate_ragged_lines=True)


def num(col):
    """Robustly coerce a column to float (strip $ , % and thousands separators)."""
    return pl.col(col).cast(pl.String).str.replace_all(r"[$,%]", "").cast(pl.Float64, strict=False)


def scatter_time(df, date_col, y_col, color=None, log_y=False, title=""):
    d = df.with_columns(
        num(y_col).alias("_y"),
        pl.col(date_col).cast(pl.String).str.to_date(strict=False).alias("_d"),
    ).drop_nulls(["_d", "_y"])
    return px.scatter(d, x="_d", y="_y", color=color, log_y=log_y,
                      labels={"_d": date_col, "_y": y_col}, title=title)


def bar_top(df, cat, val, n=20, title=""):
    d = df.with_columns(num(val).alias("_v")).drop_nulls("_v").sort("_v", descending=True).head(n)
    return px.bar(d, x=cat, y="_v", labels={"_v": val}, title=title)

## AI Models — training compute over time

In [2]:
scatter_time(load("ai_models__notable_ai_models.csv"), "Publication date", "Training compute (FLOP)",
             color="Domain", log_y=True, title="Notable AI models: training compute (FLOP) over time")

## ML Hardware — FP16 FLOP/s per watt

In [3]:
hw = load("ml_hardware.csv").with_columns(
    pl.col("Release date").cast(pl.String).str.to_date(strict=False).alias("_d"),
    (num("Tensor-FP16/BF16 performance (FLOP/s)") / num("TDP (W)")).alias("_fpw"),
).drop_nulls(["_d", "_fpw"])
px.scatter(hw, x="_d", y="_fpw", color="Manufacturer", log_y=True,
           labels={"_d": "Release date", "_fpw": "FP16 FLOP/s per watt"},
           title="ML hardware: energy efficiency over time")

## AI Chip Components — HBM (memory) economics over time
HBM is the on-package memory. Below: (1) HBM cost as a **proportion of each
chip's component cost**, then (2) **total HBM spend** across all chips per quarter.
Source: `ai_chip_components` (quarterly, by chip).

In [4]:
share = load("ai_chip_components__quarterly_by_chip.csv").with_columns(
    pl.col("Start date").cast(pl.String).str.to_date(strict=False).alias("_d"),
    num("HBM share (%) (median)").alias("_share"),
).drop_nulls(["_d", "_share"]).sort("_d")
px.line(share, x="_d", y="_share", color="Chip type", markers=True,
        labels={"_d": "Quarter", "_share": "HBM cost as % of chip cost"},
        title="HBM (memory) cost as a share of chip cost over time")

### Average HBM cost per chip over time
Total HBM cost (USD, median) divided by the number of chips modeled each quarter,
which normalizes out how many chips are in the panel.

Caveat: Q1 2026 is still understated — the chips missing from that latest quarter
(NVIDIA and 'Other') are the high-HBM ones, so the *mix* is lower, not just the count.

In [5]:
per_chip = (
    load("ai_chip_components__quarterly_by_chip.csv")
    .with_columns(
        pl.col("Start date").cast(pl.String).str.to_date(strict=False).alias("_d"),
        num("HBM cost (USD) (median)").alias("_hbm"),
    )
    .drop_nulls(["_d", "_hbm"])
    .group_by("_d").agg(pl.col("_hbm").mean().alias("_avg"), pl.len().alias("_n")).sort("_d")
)
px.bar(per_chip, x="_d", y="_avg", hover_data=["_n"],
       labels={"_d": "Quarter", "_avg": "Avg HBM cost per chip (USD, median)", "_n": "chips"},
       title="Average HBM (memory) cost per chip over time")

## AI Chip Sales — cumulative H100-equivalent compute

In [6]:
scatter_time(load("ai_chip_sales__cumulative_timelines.csv"), "End date", "H100e compute power (median)",
             color="Chip manufacturer", log_y=True, title="AI chip sales: cumulative H100e compute power")

## Frontier Data Centers — largest by power

In [7]:
bar_top(load("data_centers__data_centers.csv"), "Name", "Current power (MW)", n=20,
        title="Largest frontier data centers by current power (MW)")

## AI Capabilities — Epoch Capabilities Index over time

In [8]:
scatter_time(load("benchmarks__epoch_capabilities_index.csv"), "Release date", "ECI Score",
             color="Organization", title="Epoch Capabilities Index (ECI) by model release date")

## GPU Clusters — H100-equivalents over time

In [9]:
scatter_time(load("gpu_clusters.csv"), "First Operational Date", "H100 equivalents",
             color="Country", log_y=True, title="GPU clusters: H100-equivalents over time")

## Polling on AI Usage

In [10]:
poll = load("polling_on_ai_usage_mar_2026.csv")
q = poll["Question"][0]
sub = poll.filter(pl.col("Question") == q).with_columns(num("Overall").alias("pct"))
px.bar(sub, x="Response", y="pct", labels={"pct": "Overall (%)"}, title=f"Polling — {q}")

## Training Cost Trends (GitHub) — cloud cost by model

In [11]:
bar_top(load("github/training_cost_trends__model_costs.csv"), "Model", "Cloud_Cost", n=20,
        title="Estimated training cloud cost by model (top 20)")

## Catalog — every cleaned dataset
Shape (rows × cols) for each cleaned CSV, so all datasets are accounted for.

In [12]:
rows = []
for p in sorted(DATA.rglob("*.csv")):
    try:
        h = pl.read_csv(p, infer_schema_length=2000, ignore_errors=True, truncate_ragged_lines=True)
        rows.append({"file": str(p.relative_to(DATA)), "rows": h.height, "cols": h.width})
    except Exception as e:
        rows.append({"file": str(p.relative_to(DATA)), "rows": -1, "cols": -1})
cat = pl.DataFrame(rows).sort("file")
print(f"{cat.height} cleaned CSV files")
with pl.Config(tbl_rows=200, fmt_str_lengths=80):
    print(cat)

149 cleaned CSV files
shape: (149, 3)
┌───────────────────────────────────────────────────────────────────┬────────┬──────┐
│ file                                                              ┆ rows   ┆ cols │
│ ---                                                               ┆ ---    ┆ ---  │
│ str                                                               ┆ i64    ┆ i64  │
╞═══════════════════════════════════════════════════════════════════╪════════╪══════╡
│ ai_chip_components__cumulative_by_chip.csv                        ┆ 102    ┆ 24   │
│ ai_chip_components__cumulative_by_designer.csv                    ┆ 43     ┆ 23   │
│ ai_chip_components__cumulative_supply_denominators.csv            ┆ 8      ┆ 12   │
│ ai_chip_components__quarterly_by_chip.csv                         ┆ 102    ┆ 33   │
│ ai_chip_components__quarterly_by_designer.csv                     ┆ 43     ┆ 32   │
│ ai_chip_components__supply_denominators.csv                       ┆ 8      ┆ 12   │
│ ai_chip_owners